# Laya: Local Support-Ticket Decisions

A self-contained demonstration of using Laya to turn a support message into a structured queue suggestion. This notebook defines its own input and question schema; it does not import code from this repository.

## What Laya does

Laya is a local, non-autoregressive decision model. Give it a state (such as a ticket) and typed questions, and it returns structured decisions such as a `choice`, `score`, or `noul` answer. It is designed for decisions, not free-form chat or text generation.

This demo asks one `choice` question: which small support queue should review a ticket? The result is a suggestion for a person, not an automated action.

## Setup

Run these commands from the repository root. `uv sync` installs Laya and the development tools declared for this project. Register the project environment as a notebook kernel if it is not already available:

```sh
uv sync
uv run --group dev python -m ipykernel install --user --name laya-local-ai-demo --display-name "Python (laya-local-ai-demo)"
```

In VS Code, select the `Python (laya-local-ai-demo)` kernel. This notebook imports only the installed Laya SDK and Python standard library; it does not import project modules.

In [7]:
import laya
from laya import Router

print(f"Laya SDK version: {laya.__version__}")

Laya SDK version: 0.3.26


## Define the decision

A `choice` question provides a small set of distinct options, with a short description of each. Laya returns the selected label and a probability distribution. Keep option sets manageable and validate the labels on your own examples.

In [8]:
questions = {
    "queue": {
        "type": "choice",
        "instructions": "Which support queue should review this ticket?",
        "criteria": {
            "billing": "Invoices, payments, duplicate charges, or refunds",
            "technical": "Product bugs, errors, outages, or integrations",
            "account": "Login, password, access, or account settings",
            "other": "A request that does not fit the other queues",
        },
    }
}

questions

{'queue': {'type': 'choice',
  'instructions': 'Which support queue should review this ticket?',
  'criteria': {'billing': 'Invoices, payments, duplicate charges, or refunds',
   'technical': 'Product bugs, errors, outages, or integrations',
   'account': 'Login, password, access, or account settings',
   'other': 'A request that does not fit the other queues'}}}

## Prepare a synthetic ticket

This example is intentionally synthetic. Avoid pasting real customer information into a demo notebook.

In [9]:
ticket = {
    "subject": "Duplicate subscription charge",
    "body": "I was charged twice for my monthly subscription. Please check the duplicate payment.",
}

ticket

{'subject': 'Duplicate subscription charge',
 'body': 'I was charged twice for my monthly subscription. Please check the duplicate payment.'}

## Run the local decision

`Router` selects a checkpoint for the input. Its first prediction downloads the selected weights if they are not cached. After that, this direct SDK example runs inference locally without calling a hosted inference API. Local disk, memory, and compute are still required.

In [10]:
# Creating the router is lightweight; the first prediction loads the needed checkpoint.
router = Router()
result = router.predict(ticket, questions)

result

In [11]:
queue_answer = result["answers"]["queue"]

print(f"Suggested queue: {queue_answer['choice']}")
print(f"Selected checkpoint: {result['routing']['model']}")
print("Choice probabilities:")
for label, probability in queue_answer["probabilities"].items():
    print(f"  {label}: {probability:.3f}")

Suggested queue: billing
Selected checkpoint: english
Choice probabilities:
  billing: 0.971
  technical: 0.009
  account: 0.007
  other: 0.013


## Takeaways and limitations

- Typed output can be consumed directly by application code instead of parsing generated prose.
- After its first checkpoint download, the direct SDK supports local inference without hosted per-request inference.
- This is a small demonstration, not a production evaluation. A prediction may be wrong, and confidence values should not be treated as calibrated for your workload without validation.
- Keep a human in the loop for consequential decisions. Laya is not a general-purpose chatbot and will not write a natural-language reply for the ticket.

See the [Laya documentation](https://github.com/NandhaKishorM/laya) for the current API, installation details, and model limitations.